# clone-your-voice: train your voice on Colab

This notebook trains a voice model **from your own recordings** and packs it into a single
`<name>.cyvpack.zip` that you download and import on your Mac with `cyv voices import`.

**Before you start**

1. You made `dataset.zip` on your Mac with `cyv dataset prepare` (it includes the consent
   statement you typed there).
2. **Only your own voice, or a voice whose owner gave you explicit permission.** No
   impersonation, fraud or harassment. See `USAGE_POLICY.md` in the repo.
3. Google Colab's terms forbid creating deepfakes. Using this notebook does not make a use
   acceptable under those terms; you are responsible for how you use it.
4. This notebook uses Applio's command-line path only. It never starts a web UI and never
   opens a tunnel (Colab's free tier restricts both).

**How to use it:** run the cells from top to bottom (Runtime > Run all also works, but the
dataset cell will wait for you to pick a file). Training 200 epochs on 5 to 10 minutes of
audio takes roughly 20 to 60 minutes on a free T4 (to be confirmed by benchmark B1).

If Colab disconnects mid-training, reconnect, run all cells again with the **same voice name**: it
picks up from the last checkpoint saved on your Drive.


## 1. Check that a GPU is attached

Training on a CPU would take days. In the menu choose **Runtime > Change runtime type > T4 GPU**.

In [1]:
import shutil
import subprocess


def _gpu_info():
    if not shutil.which("nvidia-smi"):
        return None
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    return r.stdout.strip() if r.returncode == 0 and r.stdout.strip() else None


GPU = _gpu_info()
if GPU is None:
    raise RuntimeError(
        "No GPU found.\n"
        "  1. Runtime > Change runtime type > select 'T4 GPU' > Save, then run this cell again.\n"
        "  2. If T4 is greyed out or you are told there is no GPU available, your free GPU quota is\n"
        "     used up. It resets after a while (often within a day); come back later, or use Colab Pro."
    )
print("GPU OK:", GPU)

GPU OK: Tesla T4, 15360 MiB


## 2. Connect Google Drive and choose a voice name

Everything important is kept in your Drive so a disconnect does not lose work:

```
MyDrive/clone-your-voice/<name>/
    dataset/       put dataset.zip here (or upload it in step 4)
    checkpoints/   training checkpoints (used to resume)
    output/        the finished <name>.cyvpack.zip
```

A popup will ask you to allow Drive access. If it fails, click the folder icon on the left, then
the Drive icon, and approve access, or run this cell again.

In [2]:
# @title Voice name and Drive folders
VOICE_NAME = "sachinkoli2"  # @param {type:"string"}
# Allowed: lowercase letters, digits, "_" and "-", 1 to 32 characters (same rule as `cyv voices`).

import re
from pathlib import Path

if not re.fullmatch(r"[a-z0-9_-]{1,32}", VOICE_NAME):
    raise ValueError(
        "VOICE_NAME must match ^[a-z0-9_-]{1,32}$ (lowercase letters, digits, _ and -)."
    )

try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception as exc:  # noqa: BLE001
    raise RuntimeError(
        "Could not mount Google Drive (" + repr(exc) + ").\n"
        "Run this cell again and approve the popup. If you use several Google accounts, pick the one\n"
        "that owns the Drive you want to use. Third-party cookies must be allowed for colab.research.google.com."
    ) from exc

if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Drive mounted but /content/drive/MyDrive is missing; re-run the cell.")

DRIVE_DIR = Path("/content/drive/MyDrive/clone-your-voice") / VOICE_NAME
DRIVE_DATASET = DRIVE_DIR / "dataset"
DRIVE_CKPT = DRIVE_DIR / "checkpoints"
DRIVE_OUT = DRIVE_DIR / "output"
for d in (DRIVE_DATASET, DRIVE_CKPT, DRIVE_OUT):
    d.mkdir(parents=True, exist_ok=True)
print("Working folder on Drive:", DRIVE_DIR)

Mounted at /content/drive
Working folder on Drive: /content/drive/MyDrive/clone-your-voice/sachinkoli2


## 3. Install Applio (pinned) and download its models

* Applio is fetched at **one exact commit** (the same one the Mac app uses), not "latest".
* It is installed in its own virtual environment (`/content/applio-venv`) so Colab's preinstalled
  packages are never changed. A `constraints.txt` freezes the packages most likely to break.
* Applio's own `prerequisites` command downloads the pretrained v2 voice models, the RMVPE pitch
  model and the ContentVec embedder.

Takes about 5 to 10 minutes. After a runtime restart, run it again (it skips what already exists).

In [3]:
# @title Install Applio at the pinned commit
import hashlib
import os
import sys
import urllib.request
from pathlib import Path

APPLIO_REPO = "https://github.com/IAHispano/Applio"
APPLIO_COMMIT = (
    "c817f1f83f689ef72e0dfeb1c17ae990b9248506"  # must equal src/cyvd/engine_applio/pin.py
)
CYV_RAW = (
    "https://raw.githubusercontent.com/kolisachint/clone-your-voice/main/notebooks"  # helper files
)

APPLIO = Path("/content/Applio")
VENV = Path("/content/applio-venv")
PY = str(VENV / "bin" / "python")
CYV = Path("/content/cyv")
CYV.mkdir(exist_ok=True)


def sh(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)))
    subprocess.run([str(c) for c in cmd], cwd=cwd, check=True)


# 1) helper files from this project (cyv_colab.py = testable logic, constraints.txt = pins).
# They are imported while Google Drive is mounted, so they must match these sha256 values
# exactly. tests/test_notebook.py fails until these are updated together with the files.
HELPER_SHA256 = {
    "cyv_colab.py": "004a5af36caba47d43d31d37211d7e470ad8d96dc27d571da505011ebf2d4eb4",
    "constraints.txt": "b6d5e81f1e02c787276aab11835eb5b5c9e69db3b9970c93d00c2bb6b3176e40",
}
for fname, expected in HELPER_SHA256.items():
    local = Path(fname)
    if local.is_file():  # running from a git checkout / uploaded next to the notebook
        blob = local.read_bytes()
    else:
        with urllib.request.urlopen(f"{CYV_RAW}/{fname}", timeout=60) as resp:
            blob = resp.read()
    got = hashlib.sha256(blob).hexdigest()
    if got != expected:
        raise RuntimeError(
            f"{fname} does not match the hash this notebook expects (got {got[:12]}..., "
            f"expected {expected[:12]}...). Refusing to run it. Re-open the latest notebook "
            "from GitHub so the notebook and its helper files are the same version."
        )
    (CYV / fname).write_bytes(blob)
sys.path.insert(0, str(CYV))
import cyv_colab

marker = VENV / ".cyv_ready"
if marker.is_file() and marker.read_text().strip() == APPLIO_COMMIT:
    print("Already installed for this commit, skipping.")
else:
    # 2) Applio source at exactly APPLIO_COMMIT
    if not (APPLIO / ".git").is_dir():
        APPLIO.mkdir(exist_ok=True)
        sh(["git", "init", "-q"], cwd=APPLIO)
        sh(["git", "remote", "add", "origin", APPLIO_REPO], cwd=APPLIO)
    sh(["git", "fetch", "--depth", "1", "origin", APPLIO_COMMIT], cwd=APPLIO)
    sh(["git", "-c", "advice.detachedHead=false", "checkout", "-q", "FETCH_HEAD"], cwd=APPLIO)
    head = subprocess.run(
        ["git", "rev-parse", "HEAD"], cwd=APPLIO, capture_output=True, text=True
    ).stdout.strip()
    assert head == APPLIO_COMMIT, f"Applio is at {head}, expected {APPLIO_COMMIT}"

    # 3) Python packages in a private venv (Applio needs Python 3.12; uv downloads it if needed)
    sh(
        ["apt-get", "install", "-y", "-qq", "portaudio19-dev"]
    )  # a dependency of Applio's sounddevice
    sh([sys.executable, "-m", "pip", "install", "-q", "uv"])
    sh(["uv", "venv", VENV, "--python", "3.12", "--seed", "--allow-existing"])
    sh(
        [
            "uv",
            "pip",
            "install",
            "-q",
            "-r",
            APPLIO / "requirements-engine.txt",
            "-c",
            CYV / "constraints.txt",
            "--extra-index-url",
            "https://download.pytorch.org/whl/cu128",
            "--index-strategy",
            "unsafe-best-match",
            "--python",
            PY,
        ]
    )

    # 4) Applio's own downloader: pretrained v2 models, RMVPE and ContentVec
    sh(
        [PY, "core.py", "prerequisites", "--models", "--pretraineds-hifigan", "--no-exe"],
        cwd=APPLIO,
    )
    marker.write_text(APPLIO_COMMIT)

sh(
    [
        PY,
        "-c",
        "import torch; print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())",
    ]
)

$ git init -q
$ git remote add origin https://github.com/IAHispano/Applio
$ git fetch --depth 1 origin c817f1f83f689ef72e0dfeb1c17ae990b9248506
$ git -c advice.detachedHead=false checkout -q FETCH_HEAD
$ apt-get install -y -qq portaudio19-dev
$ /usr/bin/python3 -m pip install -q uv
$ uv venv /content/applio-venv --python 3.12 --seed --allow-existing
$ uv pip install -q -r /content/Applio/requirements-engine.txt -c /content/cyv/constraints.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match --python /content/applio-venv/bin/python
$ /content/applio-venv/bin/python core.py prerequisites --models --pretraineds-hifigan --no-exe
$ /content/applio-venv/bin/python -c import torch; print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())


## 4. Provide your dataset

Use the `dataset.zip` made by `cyv dataset prepare`. Two ways:

* **upload**: a file picker appears and the zip is also saved into your Drive folder (so a restart doesn't need a new upload);
* **drive**: use the zip already in `MyDrive/clone-your-voice/<name>/dataset/dataset.zip`.

The notebook refuses the zip if the consent statement is missing, or the audio is too short.

In [4]:
# @title Upload or select dataset.zip
DATASET_SOURCE = "upload"  # @param ["upload", "drive"]


zip_path = DRIVE_DATASET / "dataset.zip"
if DATASET_SOURCE == "upload" or not zip_path.is_file():
    from google.colab import files

    print("Choose dataset.zip ...")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file was uploaded. Run the cell again.")
    (DRIVE_DATASET / "dataset.zip").write_bytes(next(iter(uploaded.values())))
print("Using", zip_path)

try:
    info = cyv_colab.validate_dataset_zip(zip_path)
except cyv_colab.DatasetError as exc:
    raise RuntimeError(f"Dataset problem: {exc}") from exc

for w in info["warnings"]:
    print("WARNING:", w)
print(f"{len(info['wav_names'])} clips, {info['total_seconds'] / 60:.1f} minutes of audio")
print("TTS base voice:", info["meta"]["base_voice"])
print("Consent recorded by:", info["consent"]["owner_name"], "on", info["consent"]["created_at"])
print("Attestation:", info["consent"]["attestation"])

import zipfile

with zipfile.ZipFile(zip_path) as zf:
    CONSENT_BYTES = zf.read("consent.json")  # copied verbatim into the pack later

DATASET_DIR = APPLIO / "assets" / "datasets" / VOICE_NAME  # only wavs in here
cyv_colab.extract_dataset(zip_path, DATASET_DIR, CYV / f"{VOICE_NAME}_meta")
print("Unpacked to", DATASET_DIR)

Choose dataset.zip ...


Saving dataset-p2.zip to dataset-p2.zip
Using /content/drive/MyDrive/clone-your-voice/sachinkoli2/dataset/dataset.zip
2 clips, 15.4 minutes of audio
TTS base voice: am_adam
Consent recorded by: Sachin Koli on 2026-10-26T17:02:41+00:00
Attestation: This is my own voice
Unpacked to /content/Applio/assets/datasets/sachinkoli2


## 5. Training settings

* **Sample rate: 40000** (default). It is Applio's default, trains faster, and sounds the same
  for speech that comes from Kokoro, which only produces 24 kHz audio anyway. Use 48000 only if
  you want maximum fidelity for `cyv convert` on high-quality recordings; it is slower and the
  difference is small.
* **Epochs: 200.** More epochs is not automatically better (it can overfit a small dataset).
  100 to 300 is the sensible range for 5 to 10 minutes of speech.
* **Batch size: 8** fits a T4 comfortably. Lower it to 4 if you see "CUDA out of memory".
* **Save every N epochs**: a checkpoint is copied to your Drive this often (and also every 2 minutes
  while it changes), so a disconnect costs at most N epochs.

In [8]:
# @title Training settings
SAMPLE_RATE = 40000  # @param [40000, 48000] {type:"raw"}
EPOCHS = 200  # @param {type:"integer"}
BATCH_SIZE = 16  # @param {type:"slider", min:1, max:16, step:1}
SAVE_EVERY = 10  # @param {type:"slider", min:1, max:50, step:1}
EXPORT_EVEN_IF_SELFTEST_FAILS = False  # @param {type:"boolean"}

assert SAMPLE_RATE in (40000, 48000), "sample rate must be 40000 or 48000"
assert 1 <= EPOCHS <= 10000 and 1 <= BATCH_SIZE <= 50 and 1 <= SAVE_EVERY <= 100

EXP_DIR = APPLIO / "logs" / VOICE_NAME  # where Applio keeps this voice's files
TRAIN_LOG = Path("/content") / f"{VOICE_NAME}_train.log"
CORES = str(os.cpu_count() or 2)
print(
    f"{VOICE_NAME}: {SAMPLE_RATE} Hz, {EPOCHS} epochs, batch {BATCH_SIZE}, save every {SAVE_EVERY}"
)

import json

params_file = DRIVE_CKPT / "run_params.json"
if params_file.is_file():
    old = json.loads(params_file.read_text())
    if old.get("sample_rate") != SAMPLE_RATE:
        raise RuntimeError(
            f"Checkpoints on Drive were made at {old.get('sample_rate')} Hz but you chose {SAMPLE_RATE}. "
            "Use the same value to resume, or delete the checkpoints folder on Drive to start over."
        )
params_file.write_text(json.dumps({"sample_rate": SAMPLE_RATE, "epochs": EPOCHS}))

sachinkoli2: 40000 Hz, 200 epochs, batch 16, save every 10


37

## 6. Preprocess, extract, train, index

These are Applio's four command-line steps:

1. `preprocess`: cuts your audio into short training slices.
2. `extract`: computes pitch (**rmvpe**) and voice features (**contentvec**) for every slice.
3. `train`: fine-tunes Applio's **pretrained v2** model on your voice. Checkpoints are mirrored to Drive.
4. `index`: builds the retrieval index that makes the clone sound more like you.

**Resuming:** if a previous run left files on Drive, they are restored and finished steps are skipped.

Applio's `core.py` reports many failures only as text and still exits normally, so every step below
checks that its output files really exist.

In [6]:
# @title 6a. Preprocess + extract features (skipped when resuming)
state = cyv_colab.restore_from_drive(DRIVE_CKPT, EXP_DIR)
if state["checkpoint"]:
    print("Found checkpoints on Drive: resuming training later; skipping preprocess and extract.")
elif state["prepared"]:
    print("Found prepared data on Drive: skipping preprocess and extract.")
else:
    sh(
        [
            PY,
            "core.py",
            "preprocess",
            "--model-name",
            VOICE_NAME,
            "--dataset-path",
            DATASET_DIR,
            "--sample-rate",
            SAMPLE_RATE,
            "--cpu-cores",
            CORES,
        ],
        cwd=APPLIO,
    )
    slices = list((EXP_DIR / "sliced_audios").glob("*.wav"))
    if not slices:
        raise RuntimeError("Preprocess produced no audio slices. Scroll up for the error message.")
    print(len(slices), "training slices")

    sh(
        [
            PY,
            "core.py",
            "extract",
            "--model-name",
            VOICE_NAME,
            "--f0-method",
            "rmvpe",
            "--embedder-model",
            "contentvec",
            "--sample-rate",
            SAMPLE_RATE,
            "--cpu-cores",
            CORES,
            "--gpu",
            "0",
        ],
        cwd=APPLIO,
    )
    if not (EXP_DIR / "filelist.txt").is_file():
        raise RuntimeError(
            "Feature extraction did not finish (no filelist.txt). Scroll up for the error message."
        )
    cyv_colab.save_prepared(EXP_DIR, DRIVE_CKPT)
    print("Prepared data saved to Drive.")

$ /content/applio-venv/bin/python core.py preprocess --model-name sachinkoli2 --dataset-path /content/Applio/assets/datasets/sachinkoli2 --sample-rate 40000 --cpu-cores 2
418 training slices
$ /content/applio-venv/bin/python core.py extract --model-name sachinkoli2 --f0-method rmvpe --embedder-model contentvec --sample-rate 40000 --cpu-cores 2 --gpu 0
Prepared data saved to Drive.


In [ ]:
# @title 6b. Train (checkpoints are copied to Drive while it runs)
weights, done_epoch = cyv_colab.find_final_weights(EXP_DIR, VOICE_NAME)
if weights and done_epoch >= EPOCHS:
    print(f"Training already finished ({weights.name}); skipping.")
else:
    print(
        "Training... the first minutes show no progress while models load. Progress lines follow."
    )
    rc = cyv_colab.run_and_sync(
        [
            PY,
            "core.py",
            "train",
            "--model-name",
            VOICE_NAME,
            "--save-every-epoch",
            SAVE_EVERY,
            "--save-only-latest",
            "--total-epoch",
            EPOCHS,
            "--sample-rate",
            SAMPLE_RATE,
            "--batch-size",
            BATCH_SIZE,
            "--gpu",
            "0",
            "--pretrained",
            "--vocoder",
            "HiFi-GAN",
        ],
        cwd=APPLIO,
        log_path=TRAIN_LOG,
        local_dir=EXP_DIR,
        drive_dir=DRIVE_CKPT,
        interval=120,
    )
    weights, done_epoch = cyv_colab.find_final_weights(EXP_DIR, VOICE_NAME)
    if rc != 0 or not weights or done_epoch < EPOCHS:
        raise RuntimeError(
            f"Training did not finish (exit code {rc}, last model epoch {done_epoch} of {EPOCHS}).\n"
            "If it says 'CUDA out of memory', lower BATCH_SIZE in step 5. If Colab disconnected, "
            "reconnect and run all cells again with the same VOICE_NAME: it resumes from Drive."
        )
print("Final model:", weights.name, f"({weights.stat().st_size / 1e6:.0f} MB)")

Training... the first minutes show no progress while models load. Progress lines follow.
/content/applio-venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Dataset has 1 speakers.
Saved generator has 1 speakers.
Initializing the generator with 1 speakers.
Using HiFi-GAN vocoder
Using AdamW optimizer
Using Single-Scale Mel loss function
Using Float16 for training.
Starting training...
/content/applio-venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker 

In [ ]:
# @title 6c. Build the index
if cyv_colab.find_index(EXP_DIR, VOICE_NAME) is None:  # `train` normally already did this
    sh(
        [PY, "core.py", "index", "--model-name", VOICE_NAME, "--index-algorithm", "Auto"],
        cwd=APPLIO,
    )
INDEX = cyv_colab.find_index(EXP_DIR, VOICE_NAME)
if INDEX is None:
    raise RuntimeError("No index file was created. Scroll up for the error message.")
cyv_colab.sync_dir(EXP_DIR, DRIVE_CKPT)
print("Index:", INDEX.name, f"({INDEX.stat().st_size / 1e6:.0f} MB)")

## 7. Self-test: hear your model

This converts one short clip **from your own dataset** through the new model and plays both.
It also prints the last training loss lines and a PASS/FAIL verdict.

The PASS/FAIL check is only a heuristic (not silent, no NaN, sensible length). **Listen to it
yourself**: it should sound like you. A robotic or noisy result usually means too little or too noisy audio.

In [ ]:
# @title Self-test
from IPython.display import Audio, display

ref_member = cyv_colab.pick_reference_clip(zip_path)
REF_IN = Path("/content/selftest_input.wav")
REF_OUT = Path("/content/selftest_output.wav")
with zipfile.ZipFile(zip_path) as zf:
    REF_IN.write_bytes(zf.read(ref_member))
REF_OUT.unlink(missing_ok=True)

weights, final_epoch = cyv_colab.find_final_weights(EXP_DIR, VOICE_NAME)
sh(
    [
        PY,
        "core.py",
        "infer",
        "--input-path",
        REF_IN,
        "--output-path",
        REF_OUT,
        "--pth-path",
        weights,
        "--index-path",
        INDEX,
        "--pitch",
        0,
        "--f0-method",
        "rmvpe",
        "--embedder-model",
        "contentvec",
        "--export-format",
        "WAV",
    ],
    cwd=APPLIO,
)

print("\nLast training log lines:")
if TRAIN_LOG.is_file():
    for line in cyv_colab.latest_loss_lines(TRAIN_LOG.read_text(errors="replace")):
        print("  ", line)
else:
    print("   (no log in this session: training was finished earlier)")

print("\nOriginal (your recording):")
display(Audio(str(REF_IN)))
print("Converted through the new model:")
if REF_OUT.is_file():
    display(Audio(str(REF_OUT)))

import wave

with wave.open(str(REF_IN), "rb") as w:
    in_seconds = w.getnframes() / w.getframerate()
if REF_OUT.is_file():
    result = cyv_colab.analyze_wav(REF_OUT, expected_seconds=in_seconds)
else:
    result = {"checks": {"output_exists": (False, "infer produced no file")}, "passed": False}
for name, (ok, detail) in result["checks"].items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}: {detail}")
SELFTEST_PASSED = result["passed"]
print("\nSELF-TEST:", "PASS" if SELFTEST_PASSED else "FAIL")

## 8. Export the voice pack

Creates `<name>.cyvpack.zip` in `MyDrive/clone-your-voice/<name>/output/` with exactly four files:
`model.pth`, `model.index`, `voice.json` (settings + checksums) and `consent.json` (your consent
statement, unchanged). Download it from Drive in your browser, then on your Mac:

```
cyv voices import ~/Downloads/<name>.cyvpack.zip
```

In [ ]:
# @title Export <name>.cyvpack.zip
if not SELFTEST_PASSED and not EXPORT_EVEN_IF_SELFTEST_FAILS:
    raise RuntimeError(
        "The self-test failed, so nothing was exported. Listen to the audio above; if it is actually fine, "
        "tick EXPORT_EVEN_IF_SELFTEST_FAILS in step 5 and run this cell again."
    )

weights, final_epoch = cyv_colab.find_final_weights(EXP_DIR, VOICE_NAME)
voice = cyv_colab.build_voice_json(
    name=VOICE_NAME,
    engine_version=APPLIO_COMMIT,
    sample_rate=SAMPLE_RATE,
    base_voice=info["meta"]["base_voice"],
    pth_sha256=cyv_colab.sha256_file(weights),
    index_sha256=cyv_colab.sha256_file(INDEX),
    epochs=final_epoch,
    dataset_seconds=info["total_seconds"],
)
local_pack = Path("/content") / f"{VOICE_NAME}.cyvpack.zip"
cyv_colab.write_pack(
    local_pack, pth_path=weights, index_path=INDEX, voice=voice, consent_bytes=CONSENT_BYTES
)

final_pack = DRIVE_OUT / local_pack.name
shutil.copy2(local_pack, final_pack)
with zipfile.ZipFile(final_pack) as zf:
    print("Pack contents:", zf.namelist())
print(json.dumps(voice, indent=2))
print(f"\nDONE: {final_pack}  ({final_pack.stat().st_size / 1e6:.0f} MB)")
print("Download it from drive.google.com > My Drive > clone-your-voice >", VOICE_NAME, "> output")

## 9. Clean up (recommended)

Your recordings are personal data. Once the pack is safely downloaded to your Mac:

1. **Delete the dataset from Drive**: open `MyDrive/clone-your-voice/<name>/dataset/` and delete
   `dataset.zip` (or run the cell below). Also delete `checkpoints/prepared.tar`: it holds slices of your
   recordings. The rest of `checkpoints/` is only needed to resume or train more later.
2. **Empty the Drive Trash.** Deleting in Drive does not erase anything: the file moves to
   **Trash** and stays there for 30 days (still counting against your storage and still readable by you).
   Open [drive.google.com/drive/trash](https://drive.google.com/drive/trash), choose
   **Empty trash** (or right-click the file > **Delete forever**).
3. Colab's temporary disk (the copies in `/content`) is wiped when the runtime ends. To do it now:
   **Runtime > Disconnect and delete runtime**.

The cell below deletes the Drive dataset and the runtime copies for you. It only acts when you
tick the box. Remember step 2 afterwards.

In [ ]:
# @title Delete dataset (Drive + this runtime)
YES_DELETE_MY_DATASET = False  # @param {type:"boolean"}

if not YES_DELETE_MY_DATASET:
    print("Nothing deleted. Tick the box and run again once your .cyvpack.zip is downloaded.")
else:
    for p in (
        DRIVE_DATASET,
        DRIVE_CKPT / cyv_colab.PREPARED_TAR,
        DATASET_DIR,
        EXP_DIR / "sliced_audios",
        EXP_DIR / "extracted",  # feature files made by `extract`
        EXP_DIR / "f0",
        EXP_DIR / "f0_voiced",
        EXP_DIR / "filelist.txt",
        CYV / f"{VOICE_NAME}_meta",  # holds a copy of consent.json
        Path("/content/selftest_input.wav"),  # self-test audio (your voice)
        Path("/content/selftest_output.wav"),
    ):
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)
        elif p.is_file():
            p.unlink()
    DRIVE_DATASET.mkdir(exist_ok=True)
    print("Deleted. Now empty the Drive Trash: https://drive.google.com/drive/trash")